# 04 · Stage 3 — TreeSHAP explainability
Exact TreeSHAP for the selected model, additivity check, national five-domain attribution and a local district explanation (blueprint §3.3).

In [1]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.configuration import load_config, configured_path, feature_columns, zones
config = load_config()
P = lambda name: configured_path(config, name)
pd.set_option("display.max_columns", 40)

In [2]:
import joblib
from src.explain.shap_profiler import explain, domain_contributions, national_domain_shares
model = joblib.load(P("model"))
features = list(model.named_steps["imputer"].feature_columns)
master = pd.read_csv(P("master_dataset"))
modelling = master.loc[master["priority_class"].notna()].reset_index(drop=True)
modelling["priority_class"] = modelling["priority_class"].astype(int)
result = explain(model, modelling[["state", *features]])
print("Output space:", result["output_space"], "| base value:", round(result["base_value"], 4),
      "| max additivity error:", f"{result['max_additivity_error']:.2e}")

Output space: probability | base value: 0.8062 | max additivity error: 4.01e-15


In [3]:
import shap
shap.summary_plot(result["values"], pd.DataFrame(result["transformed"], columns=features), max_display=15, show=False)
plt.tight_layout(); plt.show()

In [4]:
domains = domain_contributions(result["values"], features)
national_domain_shares(domains, (modelling["priority_class"] == 1).to_numpy())

,domain,share_pct,mean_signed_contribution,scheme
0,emergency_facility_quality,27.10,-0.008595,LaQshya & NHM FRU/Blood Bank Upgradation
1,maternal_health_nutrition,27.07,-0.023015,POSHAN Abhiyaan & Anaemia Mukt Bharat
2,healthcare_access_delivery,19.58,-0.012805,PMSMA (9th of every month high-risk ANC) & Jan...
3,socioeconomic_education,14.50,-0.020393,Beti Bachao Beti Padhao & PM-JAY Ayushman Bharat
4,infrastructure_wash,11.76,-0.012803,Jal Jeevan Mission & Swachh Bharat / IPHS PHC ...


In [5]:
actions = pd.read_csv(P("priority_table"))
actions.loc[actions["priority_class"] == 1, ["rank", "district", "state", "priority_risk_score", "primary_domain_bottleneck", "top_1_driver", "top_2_driver"]].head(15)

,rank,district,state,priority_risk_score,primary_domain_bottleneck,top_1_driver,top_2_driver
0,1,Dibrugarh,Assam,70.0,emergency_facility_quality,public_c_section_pct: 19.90 (state median 13.8...,clean_cooking_fuel_pct: 44.70 (state median 40...
1,2,Balod,Chhattisgarh,70.0,emergency_facility_quality,public_c_section_pct: 13.00 (state median 7.00...,clean_cooking_fuel_pct: 33.30 (state median 25...
2,3,Rajnandgaon,Chhattisgarh,70.0,emergency_facility_quality,public_c_section_pct: 6.10 (state median 7.00;...,female_high_blood_sugar_pct: 10.30 (state medi...
3,4,Uttar Bastar Kanker,Chhattisgarh,70.0,emergency_facility_quality,hmis_c_section_rate_pct: 5.92 (state median 9....,oop_expenditure_log: 3.18 (state median 3.20; ...
4,5,Fatehabad,Haryana,70.0,emergency_facility_quality,public_c_section_pct: 12.20 (state median 11.4...,married_before_18_pct: 11.70 (state median 11....
5,6,Balaghat,Madhya Pradesh,70.0,infrastructure_wash,oop_expenditure_log: 3.15 (state median 3.19; ...,public_c_section_pct: 14.50 (state median 7.90...
6,7,Bhind,Madhya Pradesh,70.0,healthcare_access_delivery,hmis_c_section_rate_pct: 3.35 (state median 6....,oop_expenditure_log: 3.10 (state median 3.19; ...
7,8,Jabalpur,Madhya Pradesh,70.0,emergency_facility_quality,public_c_section_pct: 1.80 (state median 7.90;...,female_high_blood_sugar_pct: 11.30 (state medi...
8,9,Khargone (West Nimar),Madhya Pradesh,70.0,healthcare_access_delivery,oop_expenditure_log: 3.22 (state median 3.19; ...,public_c_section_pct: 13.60 (state median 7.90...
9,10,Neemuch,Madhya Pradesh,70.0,emergency_facility_quality,hmis_c_section_rate_pct: 4.05 (state median 6....,public_c_section_pct: 10.10 (state median 7.90...


In [6]:
top = actions.loc[actions["priority_class"] == 1].iloc[0]
i = modelling.index[(modelling["district"] == top["district"]) & (modelling["state"] == top["state"])][0]
explanation = shap.Explanation(values=result["values"][i], base_values=result["base_value"],
                               data=result["transformed"][i], feature_names=features)
shap.plots.waterfall(explanation, max_display=12, show=False)
plt.title(f"{top['district']}, {top['state']}"); plt.tight_layout(); plt.show()